# 🔍 锂电池制造缺陷分析 Notebook

## 分析内容
本 Notebook 不关注缺陷对整机电性能的影响，而是**聚焦于缺陷本身**的分析：

| 分析模块 | 内容 |
|---|---|
| **Part 1** 缺陷形貌可视化 | 孔隙率/厚度的空间分布场（2D/3D 热力图） |
| **Part 2** 缺陷严重度量化 | 不均匀性系数 σ/μ、梯度场、缺陷面积占比 |
| **Part 3** 局部响应分析 | 缺陷区域的局部电流密度、局部过电位、局部温度 |
| **Part 4** 老化风险评估 | 缺陷区域析锂风险、SEI 增长速率、活性材料损失 |
| **Part 5** 缺陷参数扫描 | 不同缺陷严重程度的影响趋势 |

> 运行前需先运行上述仿真 Cell 获取 `sol_2d_ok` 和 `sol_2d_def` 数据

In [ ]:
# ============================================================
# 环境配置 + 导入
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import time

# 中文字体
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

import pybamm
pybamm.set_logging_level("WARNING")  # 减少日志噪声

print(f"PyBaMM {pybamm.__version__} | NumPy {np.__version__} | Matplotlib {mpl.__version__}")
print("✅ 导入完成")

In [ ]:
# ============================================================
# 定义缺陷参数函数 + 运行仿真
# ============================================================

def _sigmoid(arg):
    return (1 + np.tanh(arg)) / 2

def _top_hat(arg, a, b, k=500):
    return _sigmoid(k * (arg - a)) * _sigmoid(k * (b - arg))

# ---------- 三种缺陷工况参数 ----------

def make_normal_params():
    return pybamm.ParameterValues("Ecker2015")

def make_drying_params():
    """烘干不均：负极边缘孔隙率骤降"""
    param = pybamm.ParameterValues("Ecker2015")
    Ly = param["Electrode width [m]"]
    Lz = param["Electrode height [m]"]
    def eps_n(x, y, z):
        return 0.329 * _top_hat(arg=y, a=Ly*0.02, b=Ly*0.98) * _top_hat(arg=z, a=Lz*0.02, b=Lz*0.98)
    param.update({"Negative electrode porosity": eps_n})
    return param

def make_coating_params():
    """涂布条纹 + 烘干不均复合缺陷"""
    param = pybamm.ParameterValues("Ecker2015")
    Ly = param["Electrode width [m]"]
    Lz = param["Electrode height [m]"]
    def eps_n(x, y, z):
        return 0.329 * _top_hat(arg=y, a=Ly*0.02, b=Ly*0.98) * _top_hat(arg=z, a=Lz*0.02, b=Lz*0.98)
    stripe_period = Ly / 5
    def eps_p(x, y, z):
        return 0.285 + 0.05 * np.sin(2*np.pi*y/stripe_period)**2
    param.update({"Negative electrode porosity": eps_n, "Positive electrode porosity": eps_p})
    return param

# ---------- 执行仿真（使用 SPMe 加速）----------

print("🔧 运行仿真中...")
model_2d = pybamm.lithium_ion.SPMe({"current collector": "potential pair", "dimensionality": 2})
var_pts = {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 8, "r_p": 8, "y": 8, "z": 8}
exp = pybamm.Experiment([("Discharge at 1C until 2.7 V",)], period="2 minutes")

solutions = {}
for label, param_fn in [("正常", make_normal_params), ("烘干缺陷", make_drying_params), ("复合缺陷", make_coating_params)]:
    t0 = time.time()
    sim = pybamm.Simulation(model_2d, parameter_values=param_fn(), experiment=exp, var_pts=var_pts)
    solutions[label] = sim.solve()
    print(f"  ✔ {label} ({time.time()-t0:.1f}s)")

sol_ok = solutions["正常"]
sol_dry = solutions["烘干缺陷"]
sol_mix = solutions["复合缺陷"]
print("\n✅ 仿真完成，可进入后续分析")

---
## Part 1: 缺陷形貌可视化 — 孔隙率空间分布场

**直接可视化缺陷本身的形状和严重程度**，不涉及电性能。

In [ ]:
# ============================================================
# Part 1: 缺陷形貌可视化 — 3种工况的孔隙率空间分布
# ============================================================

param_ref = make_normal_params()
Ly = param_ref["Electrode width [m]"]
Lz = param_ref["Electrode height [m]"]

# 创建空间网格用于可视化
ny, nz = 200, 200
y_grid = np.linspace(0, Ly, ny)
z_grid = np.linspace(0, Lz, nz)
Y, Z = np.meshgrid(y_grid, z_grid)

def eval_porosity(func, y_arr, z_arr, Ly_val, Lz_val):
    """在 y-z 网格上评估孔隙率函数"""
    result = np.zeros_like(y_arr)
    for i in range(y_arr.shape[0]):
        for j in range(y_arr.shape[1]):
            result[i, j] = func(0, y_arr[i, j], z_arr[i, j])
    return result

# 计算三种工况的负极孔隙率场
eps_n_ref = np.full_like(Y, 0.329)
eps_n_dry = eval_porosity(
    make_drying_params()["Negative electrode porosity"], Y, Z, Ly, Lz
)
eps_n_mix = eval_porosity(
    make_coating_params()["Negative electrode porosity"], Y, Z, Ly, Lz
)

# 正极孔隙率场（仅复合缺陷有变化）
eps_p_ref = np.full_like(Y, 0.285)
eps_p_mix = eval_porosity(
    make_coating_params()["Positive electrode porosity"], Y, Z, Ly, Lz
)

# ==================== 绘图 ====================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cmap = "RdYlGn"  # 绿=正常, 红=缺陷

datasets = [
    (axes[0,0], eps_n_ref, "正常电池 — 负极孔隙率", 0, 0.4),
    (axes[0,1], eps_n_dry, "烘干缺陷 — 负极孔隙率", 0, 0.4),
    (axes[0,2], eps_n_mix, "复合缺陷 — 负极孔隙率", 0, 0.4),
    (axes[1,0], eps_p_ref, "正常电池 — 正极孔隙率", 0.2, 0.4),
    (axes[1,1], eps_p_ref, "烘干缺陷 — 正极孔隙率", 0.2, 0.4),
    (axes[1,2], eps_p_mix, "复合缺陷 — 正极孔隙率", 0.2, 0.4),
]

for ax, data, title, vmin, vmax in datasets:
    Y_mm = Y * 1000  # 转为 mm
    Z_mm = Z * 1000
    im = ax.pcolormesh(Y_mm, Z_mm, data, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
    ax.set_xlabel("y [mm]")
    ax.set_ylabel("z [mm]")
    ax.set_title(title, fontsize=11)
    ax.set_aspect("equal")
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.05)
    plt.colorbar(im, cax=cax, label="孔隙率 ε")

fig.suptitle("缺陷形貌：孔隙率空间分布场 (y-z 截面)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

# 打印缺陷统计
print(f"\n{'工况':<12} {'ε_n范围':>18} {'ε_n均值':>10} {'ε_n标准差':>10} {'缺陷面积比':>10}")
print("-" * 65)
for label, data in [("正常", eps_n_ref), ("烘干缺陷", eps_n_dry), ("复合缺陷", eps_n_mix)]:
    rho_min, rho_max = data.min(), data.max()
    rho_mean = data.mean()
    rho_std = data.std()
    defect_ratio = np.sum(data < rho_mean * 0.9) / data.size * 100
    print(f"{label:<12} {rho_min:.4f} ~ {rho_max:.4f}  {rho_mean:>10.4f} {rho_std:>10.4f} {defect_ratio:>9.1f}%")

---
## Part 2: 缺陷严重度量化分析

量化指标：变异系数 CV、梯度场、缺陷面积占比、不均匀性指数。

In [ ]:
# ============================================================
# Part 2: 缺陷严重度量化分析
# ============================================================

from matplotlib.gridspec import GridSpec

# ---- 2a. 缺陷几何特征提取 ----
# 对烘干缺陷进行边界检测
def detect_defect_boundary(field, threshold_ratio=0.5):
    """检测缺陷边界：孔隙率低于阈值比的区域"""
    mean_val = np.max(field)  # 用最大值（正常区域）作为参考
    mask = field < mean_val * threshold_ratio
    from scipy.ndimage import binary_dilation, binary_erosion
    boundary = np.logical_xor(binary_dilation(mask, iterations=2), binary_erosion(mask, iterations=1))
    return mask, boundary

mask_dry, bnd_dry = detect_defect_boundary(eps_n_dry, threshold_ratio=0.8)
mask_mix, bnd_mix = detect_defect_boundary(eps_n_mix, threshold_ratio=0.8)

# ---- 2b. 梯度场分析 ----
def compute_gradient_magnitude(field, dy, dz):
    grad_y, grad_z = np.gradient(field, dy, dz)
    return np.sqrt(grad_y**2 + grad_z**2)

dy = Ly / ny
dz = Lz / nz
grad_dry = compute_gradient_magnitude(eps_n_dry, dy, dz)
grad_mix = compute_gradient_magnitude(eps_n_mix, dy, dz)
grad_ref = compute_gradient_magnitude(eps_n_ref, dy, dz)

# ---- 2c. 综合严重度评分 ----
def severity_metrics(field, name):
    """计算缺陷严重度综合指标"""
    mu = field.mean()
    sigma = field.std()
    cv = sigma / mu if mu > 0 else 0  # 变异系数
    grad_max = compute_gradient_magnitude(field, dy, dz).max()
    nonuniform = np.max(field) - np.min(field)  # 不均匀范围
    effective_area = np.sum(field > 0) / field.size * 100  # 有效活性面积
    defect_area = np.sum(field < mu * 0.9) / field.size * 100
    # 综合评分 (0~100, 越高越严重)
    score = min(100, (cv * 50 + defect_area * 0.3 + nonuniform * 20))
    return {
        "name": name, "mu": mu, "sigma": sigma, "CV": cv,
        "grad_max": grad_max, "nonuniformity": nonuniform,
        "effective_area%": effective_area, "defect_area%": defect_area,
        "severity_score": score
    }

metrics = [
    severity_metrics(eps_n_ref, "正常-负极"),
    severity_metrics(eps_n_dry, "烘干缺陷-负极"),
    severity_metrics(eps_n_mix, "复合缺陷-负极"),
    severity_metrics(eps_p_ref, "正常-正极"),
    severity_metrics(eps_p_mix, "复合缺陷-正极"),
]

# ---- 绘图 ----
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# (a) 缺陷边界叠加
ax1 = fig.add_subplot(gs[0, 0])
Y_mm, Z_mm = Y * 1000, Z * 1000
ax1.pcolormesh(Y_mm, Z_mm, eps_n_dry, cmap="RdYlGn", vmin=0, vmax=0.4, shading="auto")
ax1.contour(Y_mm, Z_mm, eps_n_dry, levels=[0.05, 0.1, 0.2], colors="k", linewidths=0.8)
ax1.set_title("(a) 烘干缺陷等值线叠加", fontsize=11)
ax1.set_xlabel("y [mm]"); ax1.set_ylabel("z [mm]")
ax1.set_aspect("equal")

# (b) 梯度场 — 烘干缺陷
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.pcolormesh(Y_mm, Z_mm, grad_dry, cmap="hot", shading="auto")
ax2.set_title("(b) 烘干缺陷 — 孔隙率梯度场 |∇ε|", fontsize=11)
ax2.set_xlabel("y [mm]"); ax2.set_ylabel("z [mm]")
ax2.set_aspect("equal")
plt.colorbar(im2, ax=ax2, label="|∇ε| [1/m]", shrink=0.8)

# (c) 梯度场 — 复合缺陷
ax3 = fig.add_subplot(gs[0, 2])
im3 = ax3.pcolormesh(Y_mm, Z_mm, grad_mix, cmap="hot", shading="auto")
ax3.set_title("(c) 复合缺陷 — 孔隙率梯度场 |∇ε|", fontsize=11)
ax3.set_xlabel("y [mm]"); ax3.set_ylabel("z [mm]")
ax3.set_aspect("equal")
plt.colorbar(im3, ax=ax3, label="|∇ε| [1/m]", shrink=0.8)

# (d) 严重度指标雷达图
ax4 = fig.add_subplot(gs[1, 0], polar=True)
categories = ["CV", "梯度\n最大值", "不均匀度", "缺陷面积\n百分比", "严重度\n评分"]
# 归一化到 [0, 1]
for m in metrics:
    m["norm_cv"] = m["CV"] / max(x["CV"] for x in metrics) if max(x["CV"] for x in metrics) > 0 else 0
    m["norm_grad"] = m["grad_max"] / max(x["grad_max"] for x in metrics) if max(x["grad_max"] for x in metrics) > 0 else 0
    m["norm_non"] = m["nonuniformity"] / max(x["nonuniformity"] for x in metrics) if max(x["nonuniformity"] for x in metrics) > 0 else 0
    m["norm_defect"] = m["defect_area%"] / max(x["defect_area%"] for x in metrics) if max(x["defect_area%"] for x in metrics) > 0 else 0
    m["norm_score"] = m["severity_score"] / max(x["severity_score"] for x in metrics) if max(x["severity_score"] for x in metrics) > 0 else 0

angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]
colors_radar = ["green", "red", "darkred", "blue", "purple"]
for m, c in zip(metrics[:3], colors_radar[:3]):
    values = [m["norm_cv"], m["norm_grad"], m["norm_non"], m["norm_defect"], m["norm_score"]]
    values += values[:1]
    ax4.plot(angles, values, "o-", color=c, label=m["name"], linewidth=1.5)
    ax4.fill(angles, values, alpha=0.1, color=c)
ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(categories, fontsize=8)
ax4.set_title("(d) 缺陷严重度雷达图（负极对比）", fontsize=11, pad=15)
ax4.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

# (e) 指标柱状图
ax5 = fig.add_subplot(gs[1, 1])
labels_bar = [m["name"] for m in metrics]
cv_vals = [m["CV"] for m in metrics]
x_pos = np.arange(len(labels_bar))
bars = ax5.bar(x_pos, cv_vals, color=["green", "red", "darkred", "blue", "purple"], alpha=0.7)
ax5.set_xticks(x_pos)
ax5.set_xticklabels(labels_bar, rotation=20, fontsize=8)
ax5.set_ylabel("变异系数 CV (σ/μ)")
ax5.set_title("(e) 变异系数对比", fontsize=11)
for bar, val in zip(bars, cv_vals):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f"{val:.4f}", ha="center", va="bottom", fontsize=7)

# (f) 缺陷面积占比 + 严重度评分
ax6 = fig.add_subplot(gs[1, 2])
defect_pct = [m["defect_area%"] for m in metrics]
severity_sc = [m["severity_score"] for m in metrics]
width = 0.35
ax6.bar(x_pos - width/2, defect_pct, width, label="缺陷面积%", color="orange", alpha=0.7)
ax6_twin = ax6.twinx()
ax6_twin.bar(x_pos + width/2, severity_sc, width, label="严重度评分", color="crimson", alpha=0.7)
ax6.set_xticks(x_pos)
ax6.set_xticklabels(labels_bar, rotation=20, fontsize=8)
ax6.set_ylabel("缺陷面积占比 [%]")
ax6_twin.set_ylabel("严重度评分 (0-100)")
ax6.set_title("(f) 缺陷面积 & 综合评分", fontsize=11)
lines1, labels1 = ax6.get_legend_handles_labels()
lines2, labels2 = ax6_twin.get_legend_handles_labels()
ax6.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper left")

fig.suptitle("Part 2: 缺陷严重度量化分析", fontsize=14, fontweight="bold")
plt.show()

# 打印完整指标表
print("\n" + "="*90)
print("缺陷严重度综合评估报告")
print("="*90)
header = f"{'工况':<16} {'μ(ε)':>8} {'σ(ε)':>8} {'CV':>8} {'|∇ε|_max':>10} {'不均匀度':>8} {'缺陷面积%':>10} {'严重度':>8}"
print(header)
print("-"*90)
for m in metrics:
    print(f"{m['name']:<16} {m['mu']:>8.4f} {m['sigma']:>8.4f} {m['CV']:>8.4f} "
          f"{m['grad_max']:>10.2f} {m['nonuniformity']:>8.4f} {m['defect_area%']:>9.1f}% {m['severity_score']:>8.1f}")
print("="*90)

---
## Part 3: 局部响应分析 — 缺陷对局部电化学行为的影响

从仿真结果中提取: 局部电流密度、局部过电位、局部SOC的空间分布。

In [ ]:
# ============================================================
# Part 3: 局部响应分析 — 缺陷区域电化学行为差异
# ============================================================

# 提取半途时刻的数据进行对比
t_mid_idx = len(sol_ok.t) // 2  # 取中间时刻

def safe_extract(solution, var_name, t_idx=None):
    """安全提取变量，如果不存在返回 None"""
    try:
        var = solution[var_name]
        if t_idx is not None:
            t_val = solution.t[t_idx]
            return var(t_val)
        return var
    except (KeyError, AttributeError):
        return None

# 可用的2D输出变量
variables_2d = [
    "Current collector current density [A.m-2]",
    "Local current collector current density [A.m-2]",
    "Negative electrode SOC",
    "Positive electrode SOC",
    "X-averaged negative electrode SOC",
    "Terminal voltage [V]",
]

# 查看哪些变量可用
print("🔍 检查仿真输出变量...")
available_vars = {}
for sol_label, sol in [("正常", sol_ok), ("烘干缺陷", sol_dry), ("复合缺陷", sol_mix)]:
    available_vars[sol_label] = []
    for v in variables_2d:
        data = safe_extract(sol, v, t_mid_idx)
        if data is not None:
            available_vars[sol_label].append(v)
            print(f"  ✔ [{sol_label}] {v}: shape={np.array(data).shape}")
        else:
            print(f"  ✗ [{sol_label}] {v}: 不可用")

# 提取可用数据
print(f"\n使用时刻 t = {sol_ok.t[t_mid_idx]:.1f}s 进行分析")

# ---- 3a. 提取并绘制电压曲线对比 ----
fig_v, ax_v = plt.subplots(1, 1, figsize=(10, 4))
for label, sol in [("正常", sol_ok), ("烘干缺陷", sol_dry), ("复合缺陷", sol_mix)]:
    t_min = sol.t / 60  # 转为分钟
    V = [sol["Terminal voltage [V]"](t) for t in sol.t]
    ax_v.plot(t_min, V, label=label, linewidth=1.5)
ax_v.set_xlabel("时间 [min]")
ax_v.set_ylabel("电压 [V]")
ax_v.set_title("电压曲线对比：缺陷对放电行为的影响", fontsize=12)
ax_v.legend()
ax_v.grid(True, alpha=0.3)
fig_v.tight_layout()
plt.show()

# ---- 3b. 容量 & 能量分析 ----
print("\n" + "="*60)
print("放电性能对比")
print("="*60)
# 兼容 NumPy 2.x (trapezoid) 和 1.x (trapz)
_trapz = getattr(np, "trapezoid", None) or np.trapz

for label, sol in [("正常", sol_ok), ("烘干缺陷", sol_dry), ("复合缺陷", sol_mix)]:
    V_arr = np.array([sol["Terminal voltage [V]"](t) for t in sol.t])
    t_arr = sol.t
    I = sol["Current [A]"](sol.t[0])
    Q = I * t_arr[-1] / 3600  # Ah
    E = _trapz(V_arr * I, t_arr) / 3600  # Wh
    print(f"  {label:<10} 最终电压={V_arr[-1]:.3f}V, 容量={Q:.3f}Ah, 能量={E:.3f}Wh")

# ---- 3c. 2D场可视化 ----
fig_2d, axes_2d = plt.subplots(1, 3, figsize=(18, 5))

for idx, (label, sol) in enumerate([("正常", sol_ok), ("烘干缺陷", sol_dry), ("复合缺陷", sol_mix)]):
    cc_var = safe_extract(sol, "Current collector current density [A.m-2]", t_mid_idx)
    if cc_var is not None:
        try:
            data = np.array(cc_var)
            if data.ndim == 1:
                n_side = int(np.sqrt(len(data)))
                if n_side * n_side == len(data):
                    data = data.reshape(n_side, n_side)
            if data.ndim == 2:
                im = axes_2d[idx].pcolormesh(data, cmap="jet")
                axes_2d[idx].set_title(f"{label}\n集流体电流密度分布", fontsize=11)
                axes_2d[idx].set_xlabel("y 网格"); axes_2d[idx].set_ylabel("z 网格")
                plt.colorbar(im, ax=axes_2d[idx], label="A.m⁻²", shrink=0.8)
                axes_2d[idx].set_aspect("equal")
            else:
                axes_2d[idx].text(0.5, 0.5, f"{label}\n数据非2D\nshape={data.shape}",
                                  ha="center", va="center", transform=axes_2d[idx].transAxes)
        except Exception as e:
            axes_2d[idx].text(0.5, 0.5, f"{label}\n处理失败:\n{e}",
                              ha="center", va="center", transform=axes_2d[idx].transAxes)
    else:
        V_arr = [sol["Terminal voltage [V]"](t) for t in sol.t]
        axes_2d[idx].plot(sol.t/60, V_arr, "b-", linewidth=1.5)
        axes_2d[idx].axvline(sol.t[t_mid_idx]/60, color="r", linestyle="--", alpha=0.5)
        axes_2d[idx].set_title(f"{label}\n电压曲线 (红=分析时刻)", fontsize=11)
        axes_2d[idx].set_xlabel("时间 [min]"); axes_2d[idx].set_ylabel("电压 [V]")
        axes_2d[idx].grid(True, alpha=0.3)

fig_2d.suptitle("Part 3: 局部电化学响应分析", fontsize=13, fontweight="bold")
fig_2d.tight_layout()
plt.show()

---
## Part 4: 老化风险评估 — 缺陷区域的潜在降解机制

分析: 锂析出风险、SEI生长加速、活性物质损失指标。基于孔隙率场推导的间接评估。

In [ ]:
# ============================================================
# Part 4: 老化风险评估
# 基于缺陷孔隙率场推导老化风险指标（间接评估法）
# ============================================================

# ---- 4a. 风险模型定义 ----
# 基于文献和经验公式的简化风险指标

def tortuosity_factor(eps, bruggeman=1.5):
    """Bruggeman 曲折度: τ = ε^(1-0.5) ≈ (1/ε)^bruggman_red"""
    return eps ** (1 - bruggeman)

def effective_diffusivity(D0, eps, bruggeman=1.5):
    """有效扩散系数 D_eff = D0 * ε^1.5"""
    return D0 * eps ** bruggeman

def current_density_nonuniformity(eps_field):
    """
    低孔隙率区域局部电流密度升高（简化模型）:
    j_local ∝ 1/ε （孔隙率越低，单位面积反应越集中）
    """
    eps_safe = np.clip(eps_field, 0.01, None)
    j_local = 1.0 / eps_safe
    j_mean = np.mean(j_local)
    return j_local / j_mean  # 归一化

def lithium_plating_risk(j_normalized, eps_field, eps_nominal=0.329):
    """
    锂析出风险指标:
    - 局部电流密度过集中 → 锂析出概率↑
    - 孔隙率过低 → 扩散受阻 → 锂来不及嵌入 → 析出↑
    R_plating = (j_norm - 1) * (eps_nominal / eps)^2
    """
    eps_safe = np.clip(eps_field, 0.01, None)
    return np.maximum(0, (j_normalized - 1) * (eps_nominal / eps_safe) ** 2)

def sei_growth_risk(j_normalized, eps_field, eps_nominal=0.329):
    """
    SEI加速生长风险:
    低孔隙率 → 局部电流密度高 → SEI生长加速
    R_sei ∝ j_norm * (eps_nominal / eps)^0.5
    """
    eps_safe = np.clip(eps_field, 0.01, None)
    return j_normalized * (eps_nominal / eps_safe) ** 0.5

def active_material_loss_risk(eps_field):
    """
    活性物质脱落/损失风险:
    孔隙率异常高 → 活性材料占比低 → 容量衰减
    孔隙率异常低 → 循环应力大 → 活性材料粉化
    R_AM_loss = |ε - ε_nominal| / ε_nominal
    """
    eps_nominal = 0.329
    return np.abs(eps_field - eps_nominal) / eps_nominal

# ---- 4b. 计算各风险场 ----
eps_nominal_n = 0.329
eps_nominal_p = 0.285

# 对三种工况的负极计算风险
risk_fields = {}
for label, eps_field in [("正常-负极", eps_n_ref), ("烘干缺陷-负极", eps_n_dry), ("复合缺陷-负极", eps_n_mix)]:
    tau = tortuosity_factor(eps_field)
    D_eff = effective_diffusivity(1.0, eps_field)  # 归一化 D0=1
    j_norm = current_density_nonuniformity(eps_field)
    R_plating = lithium_plating_risk(j_norm, eps_field, eps_nominal_n)
    R_sei = sei_growth_risk(j_norm, eps_field, eps_nominal_n)
    R_am = active_material_loss_risk(eps_field)
    risk_fields[label] = {
        "τ": tau, "D_eff": D_eff, "j_norm": j_norm,
        "R_plating": R_plating, "R_sei": R_sei, "R_am": R_am,
    }

# 正极风险（仅复合缺陷）
risk_fields["复合缺陷-正极"] = {
    "τ": tortuosity_factor(eps_p_mix),
    "D_eff": effective_diffusivity(1.0, eps_p_mix),
    "j_norm": current_density_nonuniformity(eps_p_mix),
    "R_plating": lithium_plating_risk(
        current_density_nonuniformity(eps_p_mix), eps_p_mix, eps_nominal_p),
    "R_sei": sei_growth_risk(
        current_density_nonuniformity(eps_p_mix), eps_p_mix, eps_nominal_p),
    "R_am": active_material_loss_risk(eps_p_mix),
}

# ---- 4c. 可视化 ----
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
Y_mm, Z_mm = Y * 1000, Z * 1000

# (a) 锂析出风险 — 烘干缺陷
ax = axes[0, 0]
im = ax.pcolormesh(Y_mm, Z_mm, risk_fields["烘干缺陷-负极"]["R_plating"],
                   cmap="YlOrRd", shading="auto")
ax.set_title("(a) 烘干缺陷 — 锂析出风险指数", fontsize=11)
ax.set_xlabel("y [mm]"); ax.set_ylabel("z [mm]"); ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="R_plating", shrink=0.8)

# (b) 锂析出风险 — 复合缺陷
ax = axes[0, 1]
im = ax.pcolormesh(Y_mm, Z_mm, risk_fields["复合缺陷-负极"]["R_plating"],
                   cmap="YlOrRd", shading="auto")
ax.set_title("(b) 复合缺陷 — 锂析出风险指数", fontsize=11)
ax.set_xlabel("y [mm]"); ax.set_ylabel("z [mm]"); ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="R_plating", shrink=0.8)

# (c) SEI生长风险
ax = axes[0, 2]
im = ax.pcolormesh(Y_mm, Z_mm, risk_fields["复合缺陷-负极"]["R_sei"],
                   cmap="YlOrRd", shading="auto")
ax.set_title("(c) 复合缺陷 — SEI加速生长风险", fontsize=11)
ax.set_xlabel("y [mm]"); ax.set_ylabel("z [mm]"); ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="R_SEI", shrink=0.8)

# (d) 活性物质损失风险
ax = axes[1, 0]
im = ax.pcolormesh(Y_mm, Z_mm, risk_fields["复合缺陷-负极"]["R_am"],
                   cmap="PuRd", shading="auto")
ax.set_title("(d) 复合缺陷 — 活性物质损失风险", fontsize=11)
ax.set_xlabel("y [mm]"); ax.set_ylabel("z [mm]"); ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="R_AM_loss", shrink=0.8)

# (e) 曲折度分布
ax = axes[1, 1]
im = ax.pcolormesh(Y_mm, Z_mm, risk_fields["烘干缺陷-负极"]["τ"],
                   cmap="coolwarm", shading="auto", vmin=0, vmax=1)
ax.set_title("(e) 烘干缺陷 — Bruggeman曲折度 τ", fontsize=11)
ax.set_xlabel("y [mm]"); ax.set_ylabel("z [mm]"); ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="τ", shrink=0.8)

# (f) 综合老化风险等级叠加
ax = axes[1, 2]
composite_risk = (
    risk_fields["复合缺陷-负极"]["R_plating"] +
    risk_fields["复合缺陷-负极"]["R_sei"] +
    risk_fields["复合缺陷-负极"]["R_am"]
)
im = ax.pcolormesh(Y_mm, Z_mm, composite_risk, cmap="hot_r", shading="auto")
ax.set_title("(f) 复合缺陷 — 综合老化风险热力图", fontsize=11)
ax.set_xlabel("y [mm]"); ax.set_ylabel("z [mm]"); ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="综合风险指数", shrink=0.8)

fig.suptitle("Part 4: 缺陷区域老化风险评估", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

# ---- 4d. 风险汇总表 ----
print("\n" + "="*80)
print("老化风险综合评估报告")
print("="*80)
print(f"{'工况':<16} {'析出R_max':>10} {'析出R_mean':>10} {'SEI_R_max':>10} "
      f"{'AM_R_max':>10} {'综合R_max':>10} {'风险等级':>10}")
print("-"*80)
risk_levels = ["安全", "低风险", "中等风险", "高风险", "极高风险"]
for label in ["正常-负极", "烘干缺陷-负极", "复合缺陷-负极"]:
    rf = risk_fields[label]
    rp_max, rp_mean = rf["R_plating"].max(), rf["R_plating"].mean()
    rs_max = rf["R_sei"].max()
    ra_max = rf["R_am"].max()
    comp = rf["R_plating"] + rf["R_sei"] + rf["R_am"]
    comp_max = comp.max()
    level_idx = min(4, int(comp_max / 2))
    print(f"{label:<16} {rp_max:>10.4f} {rp_mean:>10.4f} {rs_max:>10.4f} "
          f"{ra_max:>10.4f} {comp_max:>10.4f} {risk_levels[level_idx]:>10}")
print("="*80)

---
## Part 5: 参数扫描 — 不同缺陷严重程度的影响趋势

扫描不同缺陷等级，绘制容量衰减、风险指标随缺陷程度变化的趋势图。

In [ ]:
# ============================================================
# Part 5: 参数扫描 — 不同缺陷严重度的影响趋势
# 通过改变缺陷参数，快速评估趋势
# ============================================================

# ---- 5a. 纯孔隙率场级别的参数扫描（不需要重新仿真）----

# 缺陷严重度等级: 通过改变边缘有效区域比例模拟
severity_levels = np.linspace(1.0, 0.0, 11)  # 1.0=正常, 0.0=完全失效

# 模拟不同程度的烘干缺陷场
def make_drying_field(severity, ny=200, nz=200):
    """生成不同严重度的烘干缺陷孔隙率场"""
    field = np.full((nz, ny), 0.329)
    Ly_vis, Lz_vis = 1.0, 1.0  # 归一化坐标
    y_lin = np.linspace(0, Ly_vis, ny)
    z_lin = np.linspace(0, Lz_vis, nz)
    Yv, Zv = np.meshgrid(y_lin, z_lin)
    # 边缘区域: 严重度越低，边缘越多区域孔隙率→0
    edge_mask_y = _top_hat(arg=Yv, a=0.02*(1-severity), b=1-0.02*(1-severity), k=500)
    edge_mask_z = _top_hat(arg=Zv, a=0.02*(1-severity), b=1-0.02*(1-severity), k=500)
    field = field * edge_mask_y * edge_mask_z
    # 确保至少有一点点孔隙率（数值稳定性）
    field = np.clip(field, 0.001, None)
    return field

# 计算不同严重度下的风险指标
scan_results = []
for sev in severity_levels:
    field = make_drying_field(sev)
    eps_safe = np.clip(field, 0.01, None)
    j_norm = 1.0 / eps_safe
    j_norm = j_norm / j_norm.mean()
    R_plating = np.maximum(0, (j_norm - 1) * (0.329 / eps_safe) ** 2)
    R_sei = j_norm * (0.329 / eps_safe) ** 0.5
    R_am = np.abs(field - 0.329) / 0.329
    D_eff_mean = (field ** 1.5).mean()
    defect_pct = np.sum(field < field.max() * 0.9) / field.size * 100

    scan_results.append({
        "severity": sev,
        "defect_pct": defect_pct,
        "eps_mean": field.mean(),
        "CV": field.std() / field.mean() if field.mean() > 0 else 0,
        "R_plating_max": R_plating.max(),
        "R_sei_max": R_sei.max(),
        "R_am_max": R_am.max(),
        "D_eff_relative": D_eff_mean / (0.329 ** 1.5),
    })

sev_arr = np.array([r["severity"] for r in scan_results])

# ---- 5b. 可视化 ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) 有效扩散系数 vs 缺陷严重度
ax = axes[0, 0]
D_eff_arr = np.array([r["D_eff_relative"] for r in scan_results])
ax.plot(sev_arr, D_eff_arr, "o-", color="steelblue", linewidth=2, markersize=6)
ax.set_xlabel("缺陷严重度 (1=正常, 0=完全失效)")
ax.set_ylabel("D_eff / D_eff,normal")
ax.set_title("(a) 有效扩散系数随缺陷严重度变化", fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(1.0, color="green", linestyle="--", alpha=0.3, label="正常基准")
ax.legend()

# (b) 各类风险指标 vs 缺陷严重度
ax = axes[0, 1]
Rp_arr = np.array([r["R_plating_max"] for r in scan_results])
Rs_arr = np.array([r["R_sei_max"] for r in scan_results])
Ra_arr = np.array([r["R_am_max"] for r in scan_results])
ax.plot(sev_arr, Rp_arr, "s-", color="red", label="锂析出风险", linewidth=2)
ax.plot(sev_arr, Rs_arr, "^-", color="orange", label="SEI生长风险", linewidth=2)
ax.plot(sev_arr, Ra_arr, "D-", color="purple", label="活性物质损失风险", linewidth=2)
ax.set_xlabel("缺陷严重度 (1=正常, 0=完全失效)")
ax.set_ylabel("风险指数 (最大值)")
ax.set_title("(b) 老化风险指标随缺陷严重度变化", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (c) 缺陷面积占比 & 变异系数
ax = axes[1, 0]
defect_arr = np.array([r["defect_pct"] for r in scan_results])
cv_arr = np.array([r["CV"] for r in scan_results])
ax2 = ax.twinx()
l1, = ax.plot(sev_arr, defect_arr, "o-", color="crimson", linewidth=2, label="缺陷面积%")
l2, = ax2.plot(sev_arr, cv_arr, "s-", color="navy", linewidth=2, label="变异系数 CV")
ax.set_xlabel("缺陷严重度")
ax.set_ylabel("缺陷面积占比 [%]", color="crimson")
ax2.set_ylabel("变异系数 CV", color="navy")
ax.set_title("(c) 缺陷面积与不均匀性", fontsize=11)
ax.legend(handles=[l1, l2], fontsize=9, loc="center right")
ax.grid(True, alpha=0.3)

# (d) 综合风险热力图 (severity × 风险类型)
ax = axes[1, 1]
risk_matrix = np.column_stack([Rp_arr, Rs_arr, Ra_arr])
im = ax.imshow(risk_matrix.T, aspect="auto", cmap="YlOrRd",
               extent=[sev_arr[0], sev_arr[-1], -0.5, 2.5])
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["锂析出", "SEI生长", "活性物质损失"])
ax.set_xlabel("缺陷严重度")
ax.set_title("(d) 风险矩阵热力图", fontsize=11)
plt.colorbar(im, ax=ax, label="风险指数")

fig.suptitle("Part 5: 参数扫描 — 缺陷严重度影响趋势", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

# ---- 5c. 打印关键阈值 ----
print("\n" + "="*70)
print("参数扫描汇总: 不同缺陷严重度下的风险评估")
print("="*70)
print(f"{'严重度':>8} {'缺陷面积%':>10} {'CV':>8} {'D_eff/D0':>10} "
      f"{'析出风险':>10} {'SEI风险':>10} {'AM风险':>10}")
print("-"*70)
for r in scan_results:
    print(f"{r['severity']:>8.2f} {r['defect_pct']:>9.1f}% {r['CV']:>8.4f} "
          f"{r['D_eff_relative']:>10.4f} {r['R_plating_max']:>10.4f} "
          f"{r['R_sei_max']:>10.4f} {r['R_am_max']:>10.4f}")
print("="*70)

# 查找临界严重度
for r in scan_results:
    if r["R_plating_max"] > 0.5:
        print(f"\n⚠️ 锂析出风险临界点: 严重度 ≈ {r['severity']:.2f} (R_plating = {r['R_plating_max']:.4f})")
        break

---
## 总结

| 分析模块 | 关键输出 |
|---------|---------|
| **Part 1: 缺陷形貌** | 孔隙率空间分布热力图，缺陷形状可视化 |
| **Part 2: 严重度量化** | CV、梯度场、缺陷面积%、综合评分、雷达图 |
| **Part 3: 局部响应** | 电压曲线对比、容量/能量损失、2D电流密度场 |
| **Part 4: 老化风险** | 锂析出/SEI/活性物质损失风险热力图与等级 |
| **Part 5: 参数扫描** | 不同缺陷严重度下各项指标变化趋势 |

> 本笔记本使用 PyBaMM SPMe 2D 模型进行快速仿真。如需更高精度，可替换为 DFN 模型。